In [1]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("dataset/product_interactions.csv")

# Display first five records
print(df.head())

# Check dataset information
print(df.info())

# Check missing values
print(df.isnull().sum())

   user_id  product_id     category product_type  price action
0        1         101  Electronics       Laptop  25000  click
1        1         102  Electronics       Laptop  30000   view
2        1         103  Electronics       Mobile  15000  click
3        1         104    Furniture        Chair   2000   view
4        2         105  Electronics       Laptop  28000  click
<class 'pandas.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   user_id       12 non-null     int64
 1   product_id    12 non-null     int64
 2   category      12 non-null     str  
 3   product_type  12 non-null     str  
 4   price         12 non-null     int64
 5   action        12 non-null     str  
dtypes: int64(3), str(3)
memory usage: 708.0 bytes
None
user_id         0
product_id      0
category        0
product_type    0
price           0
action          0
dtype: int64


In [2]:
action_weights = {
    "view": 1,
    "click": 2
}

df["action_weight"] = df["action"].map(action_weights)

print(df[["user_id", "product_id", "action", "action_weight"]])

    user_id  product_id action  action_weight
0         1         101  click              2
1         1         102   view              1
2         1         103  click              2
3         1         104   view              1
4         2         105  click              2
5         2         106   view              1
6         2         107  click              2
7         3         108   view              1
8         3         109  click              2
9         3         110   view              1
10        4         111  click              2
11        4         112   view              1


In [3]:
categorical_features = [
    "category",
    "product_type"
]

numerical_features = [
    "price",
    "action_weight"
]

print("Categorical:", categorical_features)
print("Numerical:", numerical_features)

Categorical: ['category', 'product_type']
Numerical: ['price', 'action_weight']


In [4]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

categorical_features = [
    "category",
    "product_type"
]

numerical_features = [
    "price",
    "action_weight"
]

# Create preprocessing transformer
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", StandardScaler(), numerical_features)
    ]
)

# Transform the dataset
X = preprocessor.fit_transform(
    df[categorical_features + numerical_features]
)

print("Preprocessing completed!")
print("Transformed dataset shape:", X.shape)
print(X.toarray() if hasattr(X, "toarray") else X)

Preprocessing completed!
Transformed dataset shape: (12, 10)
[[ 0.          1.          0.          0.          0.          1.
   0.          0.          1.0864463   1.        ]
 [ 0.          1.          0.          0.          0.          1.
   0.          0.          1.55440595 -1.        ]
 [ 0.          1.          0.          0.          0.          0.
   1.          0.          0.15052702  1.        ]
 [ 0.          0.          1.          1.          0.          0.
   0.          0.         -1.06616805 -1.        ]
 [ 0.          1.          0.          0.          0.          1.
   0.          0.          1.36722209  1.        ]
 [ 0.          1.          0.          0.          0.          0.
   1.          0.         -0.13024877 -1.        ]
 [ 0.          0.          1.          0.          0.          0.
   0.          1.         -0.78539227  1.        ]
 [ 0.          1.          0.          0.          0.          1.
   0.          0.          0.80567052 -1.        ]
 [ 

In [5]:
from sklearn.neighbors import NearestNeighbors

# Create KNN model
knn_model = NearestNeighbors(
    n_neighbors=3,
    metric="cosine",
    algorithm="brute"
)

# Train the model
knn_model.fit(X)

print("KNN model trained successfully!")

KNN model trained successfully!


In [6]:
# Find 3 nearest neighbors for the first interaction
distances, indices = knn_model.kneighbors(
    X[0:1]
)

print("Nearest neighbor indices:", indices)
print("Distances:", distances)

# Display matching records
print("\nSimilar interactions:")
print(df.iloc[indices[0]])

Nearest neighbor indices: [[ 0  4 10]]
Distances: [[0.         0.00582636 0.32357917]]

Similar interactions:
    user_id  product_id     category product_type  price action  action_weight
0         1         101  Electronics       Laptop  25000  click              2
4         2         105  Electronics       Laptop  28000  click              2
10        4         111  Electronics       Mobile  18000  click              2


In [11]:
def recommend_for_user(user_id, top_n=5):

    user_history = df[df["user_id"] == user_id]

    if user_history.empty:
        return pd.DataFrame()

    interacted_products = set(user_history["product_id"])

    recommendation_scores = {}

    for index in user_history.index:

        # Keep the input 2D
        query_vector = X[index:index + 1]

        distances, indices = knn_model.kneighbors(
            query_vector,
            n_neighbors=min(5, len(df))
        )

        for distance, neighbor_index in zip(
            distances[0], indices[0]
        ):

            neighbor = df.iloc[neighbor_index]
            product_id = neighbor["product_id"]

            if product_id in interacted_products:
                continue

            similarity = 1 / (1 + distance)

            score = similarity * neighbor["action_weight"]

            recommendation_scores[product_id] = (
                recommendation_scores.get(product_id, 0) + score
            )

    ranked_products = sorted(
        recommendation_scores.items(),
        key=lambda item: item[1],
        reverse=True
    )

    recommendations = []

    for product_id, score in ranked_products[:top_n]:

        product = df[df["product_id"] == product_id].iloc[0]

        recommendations.append({
            "product_id": int(product_id),
            "category": product["category"],
            "product_type": product["product_type"],
            "price": float(product["price"]),
            "recommendation_score": round(score, 4)
        })

    return pd.DataFrame(recommendations)

In [12]:
recommendations = recommend_for_user(user_id=1, top_n=5)

print("Recommended Products for User 1:")
print(recommendations)

Recommended Products for User 1:
   product_id     category product_type    price  recommendation_score
0         105  Electronics       Laptop  28000.0                4.8292
1         111  Electronics       Mobile  18000.0                3.4866
2         106  Electronics       Mobile  12000.0                1.8359
3         107    Furniture        Table   5000.0                1.1214
4         110    Furniture        Chair   2500.0                0.9998


In [13]:
def precision_at_k(recommended_products, relevant_products, k=5):
    recommended = recommended_products[:k]

    if len(recommended) == 0:
        return 0

    relevant_count = sum(
        1 for product in recommended
        if product in relevant_products
    )

    return relevant_count / len(recommended)

In [14]:
recommended_products = [105, 106, 107, 108, 109]
relevant_products = [105, 107, 110]

precision = precision_at_k(
    recommended_products,
    relevant_products,
    k=5
)

print("Precision@5:", precision)

Precision@5: 0.4


In [2]:
import joblib
import os

os.makedirs("../models", exist_ok=True)

model_bundle = {
    "model": knn_model,
    "preprocessor": preprocessor,
    "categorical_features": categorical_features,
    "numerical_features": numerical_features
}

joblib.dump(model_bundle, "../models/knn_recommendation.pkl")

print("KNN model saved successfully!")

NameError: name 'knn_model' is not defined

In [16]:
saved_model = joblib.load("../models/knn_recommendation.pkl")

print("Model:", saved_model["model"])
print("Preprocessor:", saved_model["preprocessor"])
print("Model loaded successfully!")

Model: NearestNeighbors(algorithm='brute', metric='cosine', n_neighbors=3)
Preprocessor: ColumnTransformer(transformers=[('cat', OneHotEncoder(handle_unknown='ignore'),
                                 ['category', 'product_type']),
                                ('num', StandardScaler(),
                                 ['price', 'action_weight'])])
Model loaded successfully!
